In [1]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.constr_methods import ConstrSolverMethod
from optim.constrained.ralm import RalmCfg
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem

In [2]:
methods = [
    ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    # ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O1), "o2_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O1), "o3_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O2), "o3_o2"),
]

# linear scaling of the domain
max_domain_scaling = [
     1., 2., 3., # 10. # 0.5
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 10

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])
cntr_center = np.array([-2., 0., 4.])
cntr_radius = 1.5


# the various metrics (describing the given curvature of the surface of optimization)

def euclid_metric(x1, x2, x3, scaling: float):
    metric = torch.eye(3)

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def scaled_metric(x1, x2, x3, scaling: float):
    # elements are assigned to this metric directly to preserve gradient history
    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.  # constant prevents degeneracy at origin
    metric[1, 1] = x2 ** 2 + 1.
    metric[2, 2] = x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def coupled_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.
    metric[0, 1] = 0.5 * x1 * x2
    metric[0, 2] = 0.5 * x1 * x3

    metric[1, 0] = 0.5 * x2 * x1
    metric[1, 1] = x2 ** 2 + 1.
    metric[1, 2] = 0.5 * x2 * x3

    metric[2, 0] = 0.5 * x3 * x1
    metric[2, 1] = 0.5 * x3 * x2
    metric[2, 2] = x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def asymmetric_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.
    metric[1, 1] = 0.5 * x1 ** 2 * x2 ** 2 + 1.
    metric[2, 2] = 0.25 * x1 ** 2 * x2 ** 2 * x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


metric_funcs = [
    (euclid_metric, "euclid"),
    (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    (asymmetric_metric, "asymmetric"),
]


In [3]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm
cntr_center /= max_starting_norm
cntr_radius /= max_starting_norm

In [4]:
# configure the constrained configurations
rgd_subsolver_cfg = RiemGradDescentCfg()
rtr_subsolver_cfg = RiemTrustRegionCfg()

ralm_cfg_subsolver_rgd = RalmCfg()
ralm_cfg_subsolver_rgd.subsolver_method = SubsolverMethod.RIEM_GRAD_DESCENT
ralm_cfg_subsolver_rgd.subsolver_cfg = rgd_subsolver_cfg

ralm_cfg_subsolver_rtr = RalmCfg()
ralm_cfg_subsolver_rtr.subsolver_method = SubsolverMethod.RIEM_TRUST_REGION
ralm_cfg_subsolver_rtr.subsolver_cfg = rtr_subsolver_cfg

optimizers = [
    ((ConstrSolverMethod.RALM, ralm_cfg_subsolver_rgd), "ralm_rgd"),
    ((ConstrSolverMethod.RALM, ralm_cfg_subsolver_rtr), "ralm_rtr")
]



In [5]:
# root directory to store output files inside
base_data_dir = Path("../data/constrained")

In [6]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    trials_region_center = cntr_center * scaling
    trials_region_radius = cntr_radius * scaling

    # create the problem
    cost, region_ineqs = create_problem(torch.tensor(trials_cost_center), [[(torch.tensor(trials_region_center), trials_region_radius)]])
    region_ineq = region_ineqs[0]

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, [region_ineq], [], p0, compute_mfld, optimizer_cfg)
        print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials:   0%|          | 0/10 [00:00<?, ?it/s]/home/samuel/Documents/School/UWaterloo_MASc/Courses/ECE_602/project/ece602-mfld-optim-taylor-approx/.venv/lib/python3.14/site-packages/torch/utils/_device.py:116: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)

Trials:  10%|█         | 1/10 [00:30<04:30, 30.07s/it]

RalmResult(success=True, p=tensor([-0.3431, -0.0680,  0.2102]), iters=23, history=RalmHistory(p_hist=tensor([[-0.2215, -0.1693,  0.0526],
        [-0.2857, -0.1182,  0.1324],
        [-0.3171, -0.0932,  0.1715],
        [-0.3323, -0.0810,  0.1907],
        [-0.3397, -0.0749,  0.2000],
        [-0.3432, -0.0720,  0.2047],
        [-0.3448, -0.0705,  0.2069],
        [-0.3455, -0.0697,  0.2081],
        [-0.3457, -0.0693,  0.2087],
        [-0.3457, -0.0691,  0.2090],
        [-0.3456, -0.0690,  0.2092],
        [-0.3455, -0.0688,  0.2093],
        [-0.3453, -0.0688,  0.2094],
        [-0.3451, -0.0687,  0.2095],
        [-0.3449, -0.0686,  0.2096],
        [-0.3447, -0.0685,  0.2096],
        [-0.3445, -0.0685,  0.2097],
        [-0.3442, -0.0684,  0.2098],
        [-0.3440, -0.0683,  0.2099],
        [-0.3438, -0.0682,  0.2099],
        [-0.3435, -0.0682,  0.2100],
        [-0.3433, -0.0681,  0.2101],
        [-0.3431, -0.0680,  0.2102]]), f_hist=tensor([2.5130e-02, 6.0106e-03, 1.4385e


Trials:  20%|██        | 2/10 [01:01<04:05, 30.70s/it]

RalmResult(success=True, p=tensor([-0.3431, -0.0680,  0.2102]), iters=23, history=RalmHistory(p_hist=tensor([[-0.2452, -0.1591,  0.0542],
        [-0.2973, -0.1132,  0.1332],
        [-0.3227, -0.0908,  0.1719],
        [-0.3351, -0.0798,  0.1909],
        [-0.3410, -0.0743,  0.2001],
        [-0.3438, -0.0717,  0.2047],
        [-0.3451, -0.0703,  0.2070],
        [-0.3456, -0.0697,  0.2081],
        [-0.3458, -0.0693,  0.2087],
        [-0.3457, -0.0691,  0.2090],
        [-0.3456, -0.0689,  0.2092],
        [-0.3455, -0.0688,  0.2093],
        [-0.3453, -0.0687,  0.2094],
        [-0.3451, -0.0687,  0.2095],
        [-0.3449, -0.0686,  0.2096],
        [-0.3446, -0.0685,  0.2097],
        [-0.3444, -0.0684,  0.2097],
        [-0.3442, -0.0684,  0.2098],
        [-0.3440, -0.0683,  0.2099],
        [-0.3438, -0.0682,  0.2100],
        [-0.3435, -0.0681,  0.2100],
        [-0.3433, -0.0681,  0.2101],
        [-0.3431, -0.0680,  0.2102]]), f_hist=tensor([2.1189e-02, 5.0659e-03, 1.2112e


Trials:  30%|███       | 3/10 [01:31<03:33, 30.53s/it]

...

subsolver failed: RiemTrustRegionResult(success=False, p=tensor([-0.7064, -0.0890,  0.7430]), iters=1000, history=RiemTrustRegionHistory(p_hist=tensor([[-0.7046, -0.0815,  0.7113],
        [-0.7063, -0.0887,  0.7438],
        [-0.7068, -0.0849,  0.7107],
        ...,
        [-0.7063, -0.0893,  0.7430],
        [-0.7068, -0.0870,  0.7098],
        [-0.7064, -0.0890,  0.7430]]), f_hist=tensor([19.1502, 19.1349, 19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348,
        19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348,
        19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348,
        19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348,
        19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348,
        19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348,
        19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348, 19.1473, 19.1348,
        19.1473, 19.1348, 19.1473, 19.1348, 19.1473,


Trials:  90%|█████████ | 9/10 [2:04:00<21:02, 1262.93s/it]

subsolver failed: RiemTrustRegionResult(success=False, p=tensor([-0.7263, -0.0957,  0.7365]), iters=1000, history=RiemTrustRegionHistory(p_hist=tensor([[-0.7240, -0.1092,  0.7061],
        [-0.7261, -0.0966,  0.7369],
        [-0.7269, -0.0961,  0.7036],
        ...,
        [-0.7263, -0.0957,  0.7365],
        [-0.7267, -0.0960,  0.7032],
        [-0.7263, -0.0957,  0.7365]]), f_hist=tensor([17.0330, 17.0182, 17.0296, 17.0180, 17.0299, 17.0180, 17.0298, 17.0180,
        17.0298, 17.0180, 17.0298, 17.0180, 17.0298, 17.0180, 17.0298, 17.0180,
        17.0298, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180,
        17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180,
        17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180,
        17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180,
        17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180, 17.0299, 17.0180,
        17.0299, 17.0180, 17.0299, 17.0180, 17.0299,


Trials: 100%|██████████| 10/10 [2:05:39<00:00, 753.93s/it]
Configurations: 24it [10:55:42, 1639.27s/it]

RalmResult(success=True, p=tensor([ 0.5698, -0.8710, -1.4893]), iters=23, history=RalmHistory(p_hist=tensor([[ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893],
        [ 0.5698, -0.8710, -1.4893]]), f_hist=tensor([25.4477, 25.4477, 25.4477, 25.4